In [ ]:
# IMPORTURI
import os, json, shutil, random, pathlib
from collections import defaultdict
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import time

# SEED - ca sa fie cat de cat reproductibil (pe GPU tot iese cate un rezultat usor diferit de fiecare data, nu stiu de ce)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

#TF cu GPU T4x2 kaggle
tf.keras.mixed_precision.set_global_policy("mixed_float16")

IMG_SIZE = 224
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

# epoci global, le-am tot schimbat pana am ramas la valorile astea
EPOCHS_FAZA1 = 15     
EPOCHS_FAZA2 = 15
EPOCHS_BINAR = 8     

#PATHS (separ test-train)
WORK_DIR = "/kaggle/working/dataset_unificat"
TRAIN_DIR = os.path.join(WORK_DIR, "train")
VALID_DIR = os.path.join(WORK_DIR, "valid")

In [ ]:
# cauta radacina fiecarui dataset adaugat in kaggle
# uneori apare la /kaggle/input/datasets/owner/slug, alteori direct la /kaggle/input/slug
# in functie de cum il adaugi, asa ca verific ambele variante
def gaseste_radacina_dataset(slug, owner):
    candidati = [
        f"/kaggle/input/datasets/{owner}/{slug}",
        f"/kaggle/input/{slug}",
    ]
    for cale in candidati:
        if os.path.exists(cale):
            return cale
    print(f"'{slug}' - negasit")
    for cale in candidati:
        print(f"  - {cale}")
    print("ERROR")
    return candidati[0]

# normalizez numele de foldere ca sa le pot compara (litere mici, fara _ sau -)
def norm_name(s):
    curatat = s.lower().replace("_", " ").replace("-", " ")
    return " ".join(curatat.split())


# caut recursiv un folder dupa nume normalizat, folosit mai ales la PlantDoc
def cauta_folder_dupa_nume(radacina, nume_cautat):
    tinta = norm_name(nume_cautat)
    for dirpath, dirnames, _ in os.walk(radacina):
        for d in dirnames:
            if norm_name(d) == tinta:
                return os.path.join(dirpath, d)
    return None


# toate dataseturile pe care le folosesc (unele doar partial, vezi mai jos)
RADACINA_BAZA = gaseste_radacina_dataset("new-plant-diseases-dataset", "vipoooool")
RADACINA_GRAU = gaseste_radacina_dataset("wheat-disease-dataset-small", "yasserhessein")
RADACINA_FS = gaseste_radacina_dataset("sunflower-fruits-and-leaves-dataset", "noamaanabdulazeem")
RADACINA_PLANTDOC_ROOT = gaseste_radacina_dataset("plantdoc-dataset", "nirmalsankalana")
RADACINA_INTEL = gaseste_radacina_dataset("intel-image-classification", "puneet6060")

In [ ]:
# În loc să copiez gigabytes de imagini peste tot, creez pur și asta symlink-uri din date în 
#dosarul unificat. Economisesc o tonă de spațiu pe disc si mult timp pe Kaggle
def creeaza_folder_gol(cale):
    os.makedirs(cale, exist_ok=True)

def LeagaPoze(sursa_folder, destinatie_folder):
    creeaza_folder_gol(destinatie_folder)
    for fisier in os.listdir(sursa_folder):
        tinta = os.path.join(destinatie_folder, fisier)
        if not os.path.exists(tinta):
            os.symlink(os.path.join(sursa_folder, fisier), tinta)

creeaza_folder_gol(TRAIN_DIR)
creeaza_folder_gol(VALID_DIR)

BAZA_TRAIN = f"{RADACINA_BAZA}/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train"
BAZA_VALID = f"{RADACINA_BAZA}/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/valid"

# prima data datasetul de baza, apoi retsul
if os.path.exists(BAZA_TRAIN):
    for clasa in os.listdir(BAZA_TRAIN):
        LeagaPoze(os.path.join(BAZA_TRAIN, clasa), os.path.join(TRAIN_DIR, clasa))
    for clasa in os.listdir(BAZA_VALID):
        LeagaPoze(os.path.join(BAZA_VALID, clasa), os.path.join(VALID_DIR, clasa))

In [ ]:
# scaneaza recursiv un dataset si aduna toate pozele, grupate dupa numele folderului parinte
# exclud folderele generice de tip train/valid/test ca nu sunt nume de clasa reale
def gaseste_foldere_cu_poze(radacina, exclude=("train", "valid", "validation", "test", "images")):
    fisiere_pe_clasa = defaultdict(list)
    for dirpath, dirnames, filenames in os.walk(radacina):
        poze = [f for f in filenames if f.lower().endswith((".jpg", ".jpeg", ".png"))]
        if not poze:
            continue
        nume_folder = os.path.basename(dirpath)
        if nume_folder.lower() in exclude:
            continue
        for f in poze:
            fisiere_pe_clasa[nume_folder].append(os.path.join(dirpath, f))
    return fisiere_pe_clasa

# adauga o specie noua (ex: grau, floarea-soarelui) peste dataset-ul de baza
# NOTE to self: nu uita ca asta face split-ul random.shuffle inainte de a taia valid,
# altfel iei mereu acelasi 15% si nu-i ok
def importa_specie_noua(radacina, prefix_specie, procent_valid=0.15):
    if not os.path.exists(radacina):
        print(f"LIPSESTE: {radacina} - dataset-ul nu pare adaugat")
        return
    fisiere_pe_clasa = gaseste_foldere_cu_poze(radacina)
    if not fisiere_pe_clasa:
        print(f"Nu am gasit poze utilizabile in {radacina}")
        return

    for nume_folder, cai_fisiere in fisiere_pe_clasa.items():
        nume_curat = nume_folder.strip().replace(" ", "_").replace("-", "_")
        clasa_noua = f"{prefix_specie}___{nume_curat}"

        random.shuffle(cai_fisiere)
        n_valid = max(1, int(len(cai_fisiere) * procent_valid))
        valid_cai = cai_fisiere[:n_valid]
        train_cai = cai_fisiere[n_valid:]

        dest_train = os.path.join(TRAIN_DIR, clasa_noua)
        dest_valid = os.path.join(VALID_DIR, clasa_noua)
        creeaza_folder_gol(dest_train)
        creeaza_folder_gol(dest_valid)

        for cale in train_cai:
            shutil.copy(cale, os.path.join(dest_train, os.path.basename(cale)))
        for cale in valid_cai:
            shutil.copy(cale, os.path.join(dest_valid, os.path.basename(cale)))

        print(f"{clasa_noua}: {len(train_cai)} train, {len(valid_cai)} valid")

In [ ]:
# uniformizez eticheta "healthy" si pentru floarea soarelui, ca sa nu am doua nume diferite pt aceeasi stare
for director in (TRAIN_DIR, VALID_DIR):
    cale_veche = os.path.join(director, "Sunflower___Fresh_Leaf")
    cale_noua = os.path.join(director, "Sunflower___healthy")
    if os.path.exists(cale_veche) and not os.path.exists(cale_noua):
        os.rename(cale_veche, cale_noua)

In [ ]:
# data pipeline - augmentation + loading from disk
# augumentarea agresiva scade acuratetea
# raman valorile acestea

def face_augmentare():
    return keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.15),
        layers.RandomZoom(0.15),
        layers.RandomContrast(0.2),
        layers.RandomBrightness(0.2),
        layers.RandomTranslation(0.1, 0.1),
    ], name="augmentare")

aug = face_augmentare()

# train set
train_ds_raw = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    label_mode="categorical", seed=SEED, shuffle=True,
)

# validation set
valid_ds_raw = tf.keras.utils.image_dataset_from_directory(
    VALID_DIR, image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    label_mode="categorical", seed=SEED, shuffle=False,
)

CLASE = train_ds_raw.class_names
NUM_CLASE = len(CLASE)
print(f"Numar de clase {NUM_CLASE}")

train_ds = train_ds_raw.prefetch(AUTOTUNE)
valid_ds = valid_ds_raw.prefetch(AUTOTUNE)

In [ ]:
# EfficientNetV2B0 ca backbone, e un compromis bun intre acuratete si viteza pe kaggle
# la inceput am incercat si ResNet50 dar antrena mult mai greu pt un rezultat similar
def construieste_model(num_clase):
    baza = keras.applications.EfficientNetV2B0(
        include_top=False, weights="imagenet",
        input_shape=(IMG_SIZE, IMG_SIZE, 3), pooling="avg",
    )
    baza.trainable = False  # inghetata in faza 1, o dezghet mai jos la fine-tuning

    intrare = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = aug(intrare)
    x = baza(x, training=False)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    iesire = layers.Dense(num_clase, activation="softmax", dtype="float32")(x)

    model = keras.Model(intrare, iesire)
    return model, baza

model, baza_model = construieste_model(NUM_CLASE)
model.summary()

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

# clasele nu sunt deloc echilibrate (unele au 2000 poze, altele 100), asa ca
# calculez class weights sa nu invete modelul doar clasele majoritare
numar_poze_per_clasa = [len(os.listdir(os.path.join(TRAIN_DIR, clasa))) for clasa in CLASE]
etichete_simulate = []
for idx, n in enumerate(numar_poze_per_clasa):
    etichete_simulate += [idx] * n

ponderi_raw = compute_class_weight(class_weight="balanced", classes=np.arange(NUM_CLASE), y=etichete_simulate)
# fara cap, unele ponderi explodeaza si strica antrenarea, deci le limitez
ponderi = np.minimum(ponderi_raw, PONDERE_MAXIMA)
class_weight_dict = {i: float(w) for i, w in enumerate(ponderi)}

raport_raw = float(max(ponderi_raw) / max(min(ponderi_raw), 1e-9))
raport_cap = float(max(ponderi) / max(min(ponderi), 1e-9))
print(f"Raport ponderi fara cap: {raport_raw:.1f}x  |  cu cap {PONDERE_MAXIMA}: {raport_cap:.1f}x\n")

for clasa, n, w_raw, w in zip(CLASE, numar_poze_per_clasa, ponderi_raw, ponderi):
    marker = "  [CAP aplicat]" if w_raw > PONDERE_MAXIMA else ""
    print(f"{clasa}: {n} poze, pondere {w_raw:.2f} -> {w:.2f}{marker}")

In [ ]:
# faza 1 - antrenez doar capul, backbone-ul ramane inghetat
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"],
)

callbacks_faza1 = [
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6),
    keras.callbacks.ModelCheckpoint("/kaggle/working/model_faza1.keras", save_best_only=True, monitor="val_accuracy"),
]

print(f"Faza 1: antrenez {EPOCHS_FAZA1} epoca/epoci (capul, baza inghetata)")

istoric_faza1 = model.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=EPOCHS_FAZA1,
    callbacks=callbacks_faza1,
    class_weight=class_weight_dict,
)

In [ ]:
# faza 2 - fine tuning. dezghet doar ultimele 40 de straturi, nu tot backbone-ul
# (am incercat tot backbone-ul o data, a durat mult mai mult si nu a iesit mai bine)
baza_model.trainable = True
for strat in baza_model.layers[:-40]:
    strat.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),  # LR mult mai mic aici, altfel strica ce a invatat deja
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"],
)

callbacks_faza2 = [
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-7),
    keras.callbacks.ModelCheckpoint("/kaggle/working/best_model.keras", save_best_only=True, monitor="val_accuracy"),
]

print(f"Faza 2: antrenez {EPOCHS_FAZA2} epoca/epoci (fine-tuning, ultimele 40 straturi)")

istoric_faza2 = model.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=EPOCHS_FAZA2,
    callbacks=callbacks_faza2,
    class_weight=class_weight_dict,
)

In [ ]:
from sklearn.metrics import classification_report

rezultat = model.evaluate(valid_ds)
# atentie: astea sunt poze din acelasi dataset (chiar daca-i split valid), deci acuratetea
# reala pe poze complet noi e mai mica, de aia testez separat si pe PlantDoc mai jos
print(f"Acuratete PlantVillage valid (referinta, poate fi umflata): {rezultat[1]*100:.2f}%")

y_adevarat, y_prezis = [], []
for imagini, etichete in valid_ds:
    predictii = model.predict(imagini, verbose=0)
    y_adevarat += list(np.argmax(etichete.numpy(), axis=1))
    y_prezis += list(np.argmax(predictii, axis=1))

print(classification_report(y_adevarat, y_prezis, target_names=CLASE))

eticheta_map = {str(i): nume for i, nume in enumerate(CLASE)}
with open("/kaggle/working/label_map.json", "w") as f:
    json.dump(eticheta_map, f, ensure_ascii=False, indent=2)

print("label_map.json salvat cu", len(CLASE), "clase")

In [ ]:
from PIL import Image

#Test real pe imaginile PlantDoc 
# acestea nu au fost vazute NICIODATA in timpul antrenarii sau validarii#
#spre deosebire de setul de validare PlantVillage de mai sus
#care contine practic versiuni augmentate ale foto sursa
def evalueaza_plantdoc_complet(model, clase, mapare_plantdoc, radacina_plantdoc, dim_intrare, top_k=3):
    radacina_test = os.path.join(radacina_plantdoc, "test")
    if not os.path.exists(radacina_test):
        print(f"Nu gasesc {radacina_test} - ai adaugat dataset-ul PlantDoc?")
        return None

    y_adevarat, y_top1 = [], []
    corect_in_topk = []
    per_clasa = defaultdict(lambda: [0, 0])

    for folder_plantdoc, clasa_noastra in mapare_plantdoc.items():
        if clasa_noastra not in clase:
            continue
        cale_gasita = cauta_folder_dupa_nume(radacina_test, folder_plantdoc)
        if not cale_gasita:
            continue

        index_adevarat = clase.index(clasa_noastra)
        for fisier in os.listdir(cale_gasita):
            if not fisier.lower().endswith((".jpg", ".jpeg", ".png")):
                continue
            img = Image.open(os.path.join(cale_gasita, fisier)).convert("RGB")
            img = img.resize((dim_intrare, dim_intrare), Image.LANCZOS)
            tensor = np.expand_dims(np.array(img).astype(np.float32), axis=0)
            predictie = model.predict(tensor, verbose=0)[0]

            index_top1 = int(np.argmax(predictie))
            indici_topk = np.argsort(predictie)[::-1][:top_k]

            y_adevarat.append(index_adevarat)
            y_top1.append(index_top1)
            corect_in_topk.append(bool(index_adevarat in indici_topk))

            per_clasa[clasa_noastra][1] += 1
            if index_top1 == index_adevarat:
                per_clasa[clasa_noastra][0] += 1

    if not y_adevarat:
        print("Nu am gasit poze de test PlantDoc pentru nicio clasa mapata.")
        return None

    acuratete_top1 = sum(1 for a, p in zip(y_adevarat, y_top1) if a == p) / len(y_adevarat)
    acuratete_topk = sum(corect_in_topk) / len(corect_in_topk)

    print(f"Acuratete top-1 (poze REALE, niciodata vazute): {acuratete_top1*100:.1f}%  "
          f"({len(y_adevarat)} poze, {len(per_clasa)} clase verificate)")
    print(f"Acuratete top-{top_k} (raspunsul corect printre primele {top_k}): {acuratete_topk*100:.1f}%")
    print()
    print("Detaliat pe clasa (top-1):")
    for clasa_noastra, (corecte, total) in sorted(per_clasa.items()):
        procent = 100 * corecte / total if total else 0
        print(f"  {clasa_noastra}: {corecte}/{total}  ({procent:.0f}%)")

    return {"y_adevarat": y_adevarat, "y_top1": y_top1, "clase": clase}

rezultate_plantdoc = evalueaza_plantdoc_complet(model, CLASE, MAPARE_PLANTDOC, RADACINA_PLANTDOC_ROOT, IMG_SIZE, top_k=3)

In [ ]:
# al doilea model: doar "e frunza sau nu e frunza" este un filtru rapid de pus inainte
# de clasificatorul mare, ca sa nu incerce sa clasifice orice poza aiurea ca boala de planta
IMG_SIZE_BINAR = 160
BINAR_DIR = "/kaggle/working/dataset_binar"

for split in ["train", "valid"]:
    for clasa in ["frunza", "nu_frunza"]:
        os.makedirs(os.path.join(BINAR_DIR, split, clasa), exist_ok=True)

def copiaza_esantion(lista_fisiere, folder_destinatie, procent_valid=0.15):
    random.shuffle(lista_fisiere)
    n_valid = max(1, int(len(lista_fisiere) * procent_valid))
    for i, cale_sursa in enumerate(lista_fisiere):
        split = "valid" if i < n_valid else "train"
        nume_fisier = f"{i}_{os.path.basename(cale_sursa)}"
        try:
            shutil.copy(cale_sursa, os.path.join(BINAR_DIR, split, folder_destinatie, nume_fisier))
        except Exception:
            pass  # unele fisiere sunt corupte in dataseturile astea, mai bine le sar decat sa pice tot

# iau maxim 40 poze din fiecare clasa de frunza, nu am nevoie de mii pt filtrul asta simplu
poze_frunze = []
for clasa in CLASE:
    folder_sursa = os.path.join(TRAIN_DIR, clasa)
    fisiere = os.listdir(folder_sursa)[:40]
    poze_frunze += [os.path.join(folder_sursa, f) for f in fisiere]
copiaza_esantion(poze_frunze, "frunza")
print(f"Poze frunza adunate: {len(poze_frunze)}")

INTEL_DIR = os.path.join(RADACINA_INTEL, "seg_train", "seg_train")
if not os.path.exists(INTEL_DIR):
    gasit = cauta_folder_dupa_nume(RADACINA_INTEL, "seg_train")
    if gasit:
        INTEL_DIR = gasit

# Intel dataset are poze cu munti, cladiri, mare etc si e perfect pt clasa "nu e frunza"
poze_nu_frunza = []
if os.path.exists(INTEL_DIR):
    for clasa_intel in os.listdir(INTEL_DIR):
        folder_sursa = os.path.join(INTEL_DIR, clasa_intel)
        if not os.path.isdir(folder_sursa):
            continue
        fisiere = os.listdir(folder_sursa)[:300]
        poze_nu_frunza += [os.path.join(folder_sursa, f) for f in fisiere]
    copiaza_esantion(poze_nu_frunza, "nu_frunza")
    print(f"Poze nu-frunza adunate:{len(poze_nu_frunza)}")
else:
    print(f"Nu gasesc folderul Intel la {INTEL_DIR} - verifica structura in celula de diagnostic")

In [ ]:
train_ds_binar_raw = tf.keras.utils.image_dataset_from_directory(
    os.path.join(BINAR_DIR, "train"),
    image_size=(IMG_SIZE_BINAR, IMG_SIZE_BINAR),
    batch_size=32,
    label_mode="binary",
    seed=SEED,
)
CLASE_BINAR = train_ds_binar_raw.class_names
train_ds_binar = train_ds_binar_raw.prefetch(AUTOTUNE)

valid_ds_binar = tf.keras.utils.image_dataset_from_directory(
    os.path.join(BINAR_DIR, "valid"),
    image_size=(IMG_SIZE_BINAR, IMG_SIZE_BINAR),
    batch_size=32,
    label_mode="binary",
    seed=SEED,
    shuffle=False,
).prefetch(AUTOTUNE)

print("Clase binare (in ordine):", CLASE_BINAR)

# MobileNetV2 e suficient de bun pt task-ul asta simplu si e rapid, nu are rost sa pun ceva mai greu
def construieste_model_binar():
    baza = keras.applications.MobileNetV2(
        include_top=False, weights="imagenet",
        input_shape=(IMG_SIZE_BINAR, IMG_SIZE_BINAR, 3), pooling="avg",
    )
    baza.trainable = False

    intrare = keras.Input(shape=(IMG_SIZE_BINAR, IMG_SIZE_BINAR, 3))
    x = layers.Lambda(keras.applications.mobilenet_v2.preprocess_input)(intrare)
    x = baza(x, training=False)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    iesire = layers.Dense(1, activation="sigmoid", dtype="float32")(x)
    return keras.Model(intrare, iesire)

model_binar = construieste_model_binar()
model_binar.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

print(f"Antrenez clasificator binar: {EPOCHS_BINAR} epoca/epoci")

model_binar.fit(
    train_ds_binar,
    validation_data=valid_ds_binar,
    epochs=EPOCHS_BINAR,
    callbacks=[keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True)],
)

model_binar.save("/kaggle/working/leaf_detector.keras")